<a href="https://colab.research.google.com/github/JulianeKaye/FUNDAI-Lab1/blob/main/Lab7_Intro_to_ML_Rabor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7: Supervised vs. Unsupervised Learning, Overfitting, and Cross-Validation

## Fundamentals of Artificial Intelligence

**Name:** Juliane Kaye F. Rabor  
**Course:** Computer Science specialized with Artificial Intelligence  
**Section:** BSCSAI-2  
**Date:** 10/20/26

**Dataset Source (GitHub raw URL):**


**GitHub URL:** https://github.com/JulianeKaye/FUNDAI-Lab1

## Description
This laboratory compares supervised and unsupervised learning, demonstrates overfitting through a complexity sweep, and evaluates models using k-fold cross-validation.

In [ ]:
# Step 3: Import Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    adjusted_rand_score,
)

In [ ]:
# Step 4: Load the Dataset from GitHub

GITHUB_RAW_URL = ""

df = pd.read_cvs(GITHUB_RAW_URL)

print("Dataset shape:", df.shape)
print()
print(df.head())
print()
print("Class distribution:")
print(df["species"].value_counts())

In [ ]:
# Step 5: Prepare Features and Labels

feature_columns = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width"
]

X = df[feature_columns].values
y = df["species"].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

## Part A: Supervised Learning (Labels ARE Used)

The model learns a mapping from features to known labels. Scaling is placed INSIDE the pipeline so that each cross-validation fold computes scaling statistics from its training portion only.
This prevents data leakage.

In [ ]:
# Step 7: Build and Train the Supervised Pipeline

supervised_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42))
])

supervised_pipeline.fir(X_train, y_train)

supervised_predictions = supervised_pipeline.predict(X_test)

print("SUPERVISED LEARNING RESULTS")
print("=" * 50)
print("Test accuracy:", accuracy_score(y_test, supervised_predictions))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, supervised_predictions))
print()
print(classification_report(y_test, supervised_predictions))

## Part B: Unsupervised Learning (Labels ARE NOT Used)

The species column is completely ignored. K-Means must discover group structure on its own. Afterwards, we compare the discovered clusters to the true labels only for evaluation purposes.

In [ ]:
# Step 9: Run K-Means Clustering

# NOTE: y is NOT used anywhere in this part.
X_unlabeled = df[feature_columns].value_counts

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
cluster_label = kmeans.fit_predict(X_unlabeled)

comparison = pd.crosstab(
    pd.Series(cluster_labels, name="Discovered Cluster"),
    pd.Series(y, name="True Species")
)

print("UNSUPERVISED LEARNING RESULTS")
print("=" * 50)
print("Cluster vs. True Species crosstab:")
print(comparison)
print()
print("Adjusted Rand Index (cluster agreement with true labels):")
print(adjusted_rand_score(y, cluster_labels))

In [ ]:
# Step 10: Visualize Clusters with PCA

pca = PCA(n_components=2)
X_pca = pca.fir_transform(X_unlabeled)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="viridis", edgecolor="k")
plt.title("K-Means Clusters (unsupervised)")
plt.xlabel("PC1")
plt.ylabel("PC2")

plt.subplot(1, 2, 2)
species_codes = pd.factorize(y)[0]
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=species_codes, cmap="viridis", edgecolor="k")
plt.title("True Species (for comparison only)")
plt.xlabel("PC1")

plt.tight_layout()
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)

## Part C: Complexity Sweep (Bias-Variance Demostration)

We train decision trees with increasing max_depth.

- Very small depth -> high bias -> underfitting
- Moderate depth -> sweet spot -> good generalization
- Very large depth -> high variance -> overfitting

In [ ]:
# Step 12: Run the Sweep

depth_values = list(range(1, 16))

train_scores = []
train_scores = []

for depth in depth_values:
  pipe = Pipeline([
      ("scaler", StandardScaler()),
      ("model", DecisionTreeClassifier(max_depth=depth, random_state=42))
  ])

  pipe.fit(X_train, y_train)

  train_scores.append(accuracy_score(y_train, pipe.predict(X_train)))
  test_scores.append(accuracy_score(y_test, pipe.predict(X_test)))

sweep_table